<h4 style="color:gray">Aprendizaje no supervisado >> Cluster Analysis >> K-means</h4>
<h1>Validación de agrupación</h1>
La validación de agrupación es el proceso de evaluación objetiva y cuantitativa del resultado de un grupo. Haremos esta validación aplicando índices de validación de grupos.

<h2>Índices Externos</h2>
Se trata de métodos de puntuación que utilizamos si se etiquetan los datos originales, lo que no es el caso más frecuente en este tipo de problemas. Emparejaremos una estructura de grupo con la información conocida de antemano.
<p>El índice más utilizado es el índice de rand ajustado.</p>
Índice Rand Ajustado (ARI) € [-1,1]
Para entenderlo, primero debemos definir sus componentes:
<img src="img/ri.png" width="280px" style="margin:0 auto;"/>
<ul>
    <li>a: es el número de puntos que están en el mismo cluster tanto en C como en K</li>
    <li>b: es el número de puntos que se encuentran en los diferentes clusters tanto en C como en K.</li>
    <li>n = es el número total de muestras</li>
</ul>
<img src="img/plot1.png" width="280px" style="margin:0 auto;"/>
<img src="img/ari.png" width="280px" style="margin:0 auto"/>
El ARI puede obtener valores que van de -1 a 1. Cuanto más alto sea el valor, mejor se ajusta a los datos originales.

<h2>Índices de Validación Interna</h2>
En el aprendizaje no supervisado, trabajaremos con datos no etiquetados y es entonces cuando los índices internos son más útiles. Uno de los índices más comunes es el Coeficiente de Silueta.<br/>
Hay un Coeficiente de Silueta para cada punto de datos.<br/>
<img src="img/silueta.png" width="190px" style="margin:0 auto"/>
<ul>
    <li>a = distancia media a otra muestra i en el mismo conglomerado</li>
    <li>b = distancia media a otra muestra i en el conglomerado vecino más cercano</li>
</ul>
El Coeficiente de Silueta (CS) puede obtener valores de -1 a 1. Cuanto más alto sea el valor, mejor será la K seleccionada. Penaliza más si superábamos la K ideal que si nos quedábamos cortos.
<p>Sólo es adecuado para ciertos algoritmos como K-Medias y agrupación jerárquica. No es conveniente utilizarlo con DBSCAN, utilizaremos DBCV en su lugar.</p>

<img src="./img/OIP.jpg" style="width:100%;height:120px;opacity:0.8"/>
<h1 style="margin:3px">K-means</h1>
Ahora que tiene una comprensión básica del agrupamiento de k-means en Python, es hora de realizar el agrupamiento de k-means en un conjunto de datos del mundo real. Estos datos contienen valores de expresión génica de un manuscrito escrito por los investigadores del proyecto de análisis <a href="https://www.cancer.gov/ccg/research/genome-sequencing/tcga" style="color:purple">Pan-Cancer de The Cancer Genome Atlas (TCGA)</a>.
<p>Hay 881 muestras (filas) que representan cinco subtipos de cáncer distintos. Cada muestra tiene valores de expresión génica para 20 531 genes (columnas). El <a href="https://archive.ics.uci.edu/ml/machine-learning-databases/00401/TCGA-PANCAN-HiSeq-801x20531.tar.gz" style="color:purple">dataset</a> está disponible en el Repositorio de aprendizaje automático de <a href="https://archive.ics.uci.edu/ml/index.php" style="color:purple">UC Irvine</a>.</p>

<h2>Creación de una tubería de agrupamiento con K-Means</h2>
<h3>Importar las librerías</h3>

In [ ]:
import tarfile
import urllib

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

<h3>Extracción del repositorio</h3>
El siguiente código descarga y lee el dataset directamente del archivo comprimido .tar.gz<br/>
Otra opción es descargar el repositorio manualmente, descomprimirlo y usar pd.<span style="color:green">read_csv</span>() 

In [ ]:
uci_tcga_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00401/"
)
archive_name = "TCGA-PANCAN-HiSeq-801x20531.tar.gz"

# build the url
full_download_url = urllib.parse.urljoin(uci_tcga_url, archive_name)

# download the file
r = urllib.request.urlretrieve(full_download_url, archive_name)

# extract the data from the archive
tar = tarfile.open(archive_name, "r:gz")
tar.extractall()
tar.close()

In [ ]:
datafile = "TCGA-PANCAN-HiSeq-801x20531/data.csv"
labels_file = "TCGA-PANCAN-HiSeq-801x20531/labels.csv"
data = np.genfromtxt(
    datafile, delimiter=",", usecols=range(1, 20532), skip_header=1
)
true_label_names = np.genfromtxt(
    labels_file, delimiter=",", usecols=(1,), skip_header=1, dtype=str
)

In [ ]:
data[:5, :3]

In [ ]:
true_label_names[:5]

<p>Las variables del dataset contiene todos los valores de expresión génica de 20.531 genes. Los true_label_names son los tipos de cáncer para cada una de las 881 muestras. El primer registro en data corresponde con la primera etiqueta en true_labels.</p>
<p>Las etiquetas son cadenas que contienen abreviaturas de tipos de cáncer:
<ul>
    <li><b>BRCA</b>: Carcinoma invasivo de mama (breast)</li>
    <li><b>COAD</b>: adenocarcinoma de colon</li>
    <li><b>KIRC</b>: Carcinoma renal de células claras de riñón (kidney)</li>
    <li><b>LUAD</b>: adenocarcinoma de pulmón (lung)</li>
    <li><b>PRAD</b>: adenocarcinoma de próstata (prostate)</li>
</ul>
Para usar estas etiquetas en los métodos de evaluación, primero debe convertir las abreviaturas a números enteros con LabelEncoder:

In [ ]:
#Codificación de categorias a números
label_encoder = LabelEncoder()
true_labels = label_encoder.fit_transform(true_label_names)
true_labels[:5]

Como label_encoder se ha ajustado a los datos, puede ver las clases únicas representadas mediante <span style="color:green">.classes_</span>. Almacene la longitud de la matriz en la variable <span style="color:green">n_clusters</span> para su uso posterior:

In [ ]:
label_encoder.classes_

In [ ]:
n_clusters = len(label_encoder.classes_)

<p>Las técnicas de <span style="color:red">reducción de la dimensionalidad</span> ayudan a abordar un problema con los algoritmos de aprendizaje automático conocido como la maldición de la dimensionalidad. En resumen, a medida que aumenta el número de características, el espacio de características se vuelve escaso. Esta escasez dificulta que los algoritmos encuentren objetos de datos cerca uno del otro en un espacio de mayor dimensión. Dado que el conjunto de datos de expresión génica tiene más de 20 000 características, califica como un gran candidato para la reducción de la dimensionalidad.</p>
<p>El <span style="color:red">análisis de componentes principales (PCA)</span> transforma los datos de entrada proyectándolos en un número menor de dimensiones llamadas componentes. Los componentes capturan la variabilidad de los datos de entrada a través de una combinación lineal de las características de los datos de entrada.</p>

In [ ]:
preprocessor = Pipeline(
    [
        ("scaler", MinMaxScaler()),
        ("pca", PCA(n_components=2, random_state=42)),
    ]
)

Un <span style="color:red">pipeline</span> (tubería) es un conjunto de fases en las que cada salida de una fase es la entrada de la siguiente.

In [ ]:
clusterer = Pipeline(
    [
        (
            "kmeans",
            KMeans(
                n_clusters=n_clusters,
                init="k-means++",
                n_init=50,
                max_iter=500,
                random_state=42,
            ),
        ),
    ]
)

In [ ]:
pipe = Pipeline([("preprocessor", preprocessor), ("clusterer", clusterer)])

Llamar a .<span style="color:green">fit</span>() con datos como argumento realiza todos los pasos de canalización en los datos:

In [ ]:
pipe.fit(data)

Podemos consultar como quedan los datos despues del preprocesamiento

In [ ]:
preprocessed_data = pipe["preprocessor"].transform(data)
preprocessed_data

Podemos consultar el resultado del agrupamiento con K-means

In [ ]:
predicted_labels = pipe["clusterer"]["kmeans"].labels_
predicted_labels

In [ ]:
pipe["clusterer"]["kmeans"].inertia_

Comparamos la agrupación original con la agrupación de K-means

In [ ]:
# Clasificación con el modelo kmeans
# ==============================================================================
y_predict = pipe["clusterer"]["kmeans"].labels_

# Representación gráfica: grupos originales vs clusters creados
# ==============================================================================
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Grupos originales
for i in np.unique(true_labels):
    ax[0].scatter(
        x = preprocessed_data[true_labels == i, 0],
        y = preprocessed_data[true_labels == i, 1], 
        c = plt.rcParams['axes.prop_cycle'].by_key()['color'][i],
        marker    = 'o',
        edgecolor = 'black', 
        label= label_encoder.classes_[i]  
    )
ax[0].set_title('Clusters generados por Kmeans')
ax[0].legend();

for i in np.unique(pipe["clusterer"]["kmeans"].labels_):
    ax[1].scatter(
        x = preprocessed_data[pipe["clusterer"]["kmeans"].labels_ == i, 0],
        y = preprocessed_data[pipe["clusterer"]["kmeans"].labels_ == i, 1], 
        c = plt.rcParams['axes.prop_cycle'].by_key()['color'][i],
        marker    = 'o',
        edgecolor = 'black', 
        label= f"Cluster {i}"
    )
    
ax[1].scatter(
    x = pipe["clusterer"]["kmeans"].cluster_centers_[:, 0],
    y = pipe["clusterer"]["kmeans"].cluster_centers_[:, 1], 
    c = 'black',
    s = 200,
    marker = '*',
    label  = 'centroides'
)
ax[1].set_title('Clusters generados por Kmeans')
ax[1].legend();

In [ ]:
# Matriz de confusión: grupos originales vs clusters creados
# ==============================================================================
pd.crosstab(true_labels, pipe["clusterer"]["kmeans"].labels_, dropna=False, rownames=['grupo_real'], colnames=['cluster'])

La agrupación parece muy buena, pero se puede mejorar.La ventaja de usar un pipeline es que basta con modificar los parámetros de entrada para que los resultados se ajusten más a lo que esperaba.
<p>El proceso de ajuste de parámetros consiste en alterar secuencialmente uno de los valores de entrada de los parámetros del algoritmo y registrar los resultados. Al final del proceso de ajuste de parámetros, tendrá un conjunto de puntuaciones de rendimiento, una para cada nuevo valor de un parámetro determinado. El ajuste de parámetros es un método eficaz para maximizar el rendimiento de su canalización de clústeres.</p>
<p>Al configurar el parámetro PCA n_components=2, comprimió todas las características en dos componentes o dimensiones. Este valor era conveniente para la visualización en un gráfico bidimensional. Pero usar solo dos componentes significa que el paso PCA no capturará toda la varianza explicada de los datos de entrada.</p>
<p>La varianza explicada mide la discrepancia entre los datos transformados por PCA y los datos de entrada reales. La relación entre n_components y la varianza explicada se puede visualizar en un gráfico para mostrarle cuántos componentes necesita en su PCA para capturar un cierto porcentaje de la varianza en los datos de entrada. También puede utilizar <span style="color:green">métricas de rendimiento de agrupación en clústeres</span> para evaluar cuántos componentes PCA son necesarios para lograr resultados de agrupación en clústeres satisfactorios.</p>
<p>La clase Pipeline es poderosa en esta situación. Le permite realizar ajustes de parámetros básicos utilizando un bucle for.</p>

In [ ]:
# Empty lists to hold evaluation metrics
silhouette_scores = []
ari_scores = []
for n in range(2, 11):
    # This set the number of components for pca,
    # but leaves other steps unchanged
    pipe["preprocessor"]["pca"].n_components = n
    pipe.fit(data)
    
    silhouette_coef = silhouette_score(
        pipe["preprocessor"].transform(data),
        pipe["clusterer"]["kmeans"].labels_,
    )
    ari = adjusted_rand_score(
        true_labels,
        pipe["clusterer"]["kmeans"].labels_,
    )

    # Add metrics to their lists
    silhouette_scores.append(silhouette_coef)
    ari_scores.append(ari)

In [ ]:
plt.style.use("fivethirtyeight")
plt.figure(figsize=(6, 6))
plt.plot(
    range(2, 11),
    silhouette_scores,
    c="#008fd5",
    label="Silhouette Coefficient",
)
plt.plot(range(2, 11), ari_scores, c="#fc4f30", label="ARI")
#plt.plot(range(2, 11), inertias, marker='o',label="elbow")

plt.xlabel("n_components")
plt.legend()
plt.title("Clustering Performance\nas a Function of n_components")
plt.tight_layout()
plt.show()

Hay dos conclusiones de este gráfico:
<p>El coeficiente de silueta disminuye linealmente. El coeficiente de silueta depende de la distancia entre los puntos, por lo que a medida que aumenta el número de dimensiones, aumenta el esparcimiento de los puntos.</p>
<p>ARI mejora significativamente a medida que agregan componentes. Parece que comienza a disminuir después de n_components=7, por lo que ese sería el valor que se usaría para presentar los mejores resultados de agrupación en clústeres de esta canalización.</p>
<p>Como la mayoría de las decisiones de aprendizaje automático, debe equilibrar la optimización de las métricas de evaluación de la agrupación en clústeres con el objetivo de la tarea de agrupación en clústeres. En situaciones en las que las etiquetas de grupos están disponibles, como en este caso, ARI es una opción razonable. ARI cuantifica la precisión con la que su canalización pudo reasignar las etiquetas de clúster.</p>
<p>El coeficiente de silueta y el método elbow, por otro lado, son una buena opción para el agrupamiento exploratorio porque ayudan a identificar subclusters.</p>

Veamos que tanto mejora la agrupación al usar 7 componentes principales usando la matriz de confusión

In [ ]:
pipe["preprocessor"]["pca"].n_components = 7
pipe.fit(data)

In [ ]:
# Matriz de confusión: grupos originales vs clusters creados
# ==============================================================================
pd.crosstab(true_labels, pipe["clusterer"]["kmeans"].labels_, dropna=False, rownames=['grupo_real'], colnames=['cluster'])

Efectivamente al usar 7 componentes principales mejoro la la clasificación de todos los grupos, de hecho en el grupo 4 ya no tuvo ningun error y en los grupos 2 y 3 solo se equivoco con 1 dato.